# Core Repositories Testing

This notebook tests the core domain repositories that act as foundational data for the system.
It covers CRUD operations and specific queries using the **Unit of Work** pattern.

In [3]:
# Setup database and import required modules
from domain.database import Base, engine
from domain.config import DB_URL
from domain.repositories.unit_of_work import UnitOfWorkFactory

from domain.models import Patient, Staff, SampleType, Protocol, Container
from domain.repositories.patient_repository import PatientRepository
from domain.repositories.staff_repository import StaffRepository
from domain.repositories.sample_type_repository import SampleTypeRepository
from domain.repositories.protocol_repository import ProtocolRepository
from domain.repositories.base_repository import BaseRepository

from datetime import date
import warnings
warnings.filterwarnings('ignore')


# Initialize Unit of Work Factory
uow_factory = UnitOfWorkFactory(DB_URL)

## 1. Patient Repository Tests

In [4]:
# Test CRUD and specific queries for PatientRepository
with uow_factory.create() as uow:
    patient_repo = PatientRepository(uow.session)

    # --- CREATE ---
    print("--- CREATE PATIENTS ---")
    p1 = Patient(code="P-001", name="John", lastname="Doe", birth_date=date(1990, 5, 14), active=True, test="Blood Test")
    p2 = Patient(code="P-002", name="Jane", lastname="Smith", birth_date=date(1985, 10, 2), active=False, test="Genetic")
    
    # Save registers the object in the session, commit persists it
    patient_repo.save(p1)
    patient_repo.save(p2)
    uow.commit()
    print(f"Created: {p1.code} -> {p1.name} {p1.lastname} (ID: {p1.id})")
    print(f"Created: {p2.code} -> {p2.name} {p2.lastname} (ID: {p2.id})\n")

    # --- READ BY ID & GET ALL ---
    print("--- READ OPERATIONS ---")
    p_read = patient_repo.get_by_id(p1.id)
    print(f"get_by_id({p1.id}): {p_read.name} {p_read.lastname}")
    
    all_patients = patient_repo.get_all()
    print(f"get_all(): Found {len(all_patients)} patients total.\n")

    # --- SPECIFIC QUERIES ---
    print("--- SPECIFIC QUERIES ---")
    found_by_code = patient_repo.get_by_code("P-002")
    print(f"get_by_code('P-002'): {found_by_code.name} {found_by_code.lastname}")
    
    active_patients = patient_repo.get_active()
    print(f"get_active(): Found {len(active_patients)} active patient(s).")
    
    p_with_samples = patient_repo.get_with_samples("P-001")
    print(f"get_with_samples('P-001'): Eagerly loaded {len(p_with_samples.samples)} samples.\n")

    # --- UPDATE ---
    print("--- UPDATE ---")
    p1.name = "Jonathan"
    uow.commit()  # Session tracks the modification
    print(f"Updated P-001 name to: {patient_repo.get_by_id(p1.id).name}\n")

    # --- DELETE ---
    print("--- DELETE ---")
    patient_repo.delete(p2)
    uow.commit()
    print(f"Deleted patient P-002. Total count is now: {patient_repo.count()}\n")

--- CREATE PATIENTS ---
Created: P-001 -> John Doe (ID: 1)
Created: P-002 -> Jane Smith (ID: 2)

--- READ OPERATIONS ---
get_by_id(1): John Doe
get_all(): Found 2 patients total.

--- SPECIFIC QUERIES ---
get_by_code('P-002'): Jane Smith
get_active(): Found 1 active patient(s).
get_with_samples('P-001'): Eagerly loaded 0 samples.

--- UPDATE ---
Updated P-001 name to: Jonathan

--- DELETE ---
Deleted patient P-002. Total count is now: 1



## 2. Staff Repository Tests

In [5]:
# Test CRUD and specific queries for StaffRepository
with uow_factory.create() as uow:
    staff_repo = StaffRepository(uow.session)

    print("--- CREATE STAFF ---")
    s1 = Staff(code="S-001", name="Alice", lastname="Wonder", role="researcher", active=True)
    s2 = Staff(code="S-002", name="Bob", lastname="Builder", role="technician", active=True)
    
    staff_repo.save(s1)
    staff_repo.save(s2)
    uow.commit()
    print(f"Created: {s1.code} ({s1.role})")
    print(f"Created: {s2.code} ({s2.role})\n")

    print("--- QUERY STAFF ---")
    s_read = staff_repo.get_by_code("S-001")
    print(f"get_by_code('S-001'): {s_read.name} {s_read.lastname}")
    print(f"Total Staff Count: {staff_repo.count()}\n")

--- CREATE STAFF ---
Created: S-001 (researcher)
Created: S-002 (technician)

--- QUERY STAFF ---
get_by_code('S-001'): Alice Wonder
Total Staff Count: 2



## 3. SampleType & Container Tests

In [6]:
# Test SampleTypeRepository and basic Container operations
with uow_factory.create() as uow:
    type_repo = SampleTypeRepository(uow.session)
    # ContainerRepository is missing in the project structure, using BaseRepository as fallback
    container_repo = BaseRepository[Container, int](uow.session, Container)

    print("--- CREATE TYPES & CONTAINERS ---")
    st1 = SampleType(type_name="Blood")
    st2 = SampleType(type_name="Saliva")
    type_repo.save(st1)
    type_repo.save(st2)
    
    c1 = Container(code="C-001", type_name="Cryotube")
    container_repo.save(c1)
    uow.commit()
    
    print(f"Created Types: {st1.type_name}, {st2.type_name}")
    print(f"Created Container: {c1.code} ({c1.type_name})\n")

    print("--- QUERY TYPES ---")
    blood_type = type_repo.get_by_name("Blood")
    print(f"get_by_name('Blood'): ID {blood_type.id}")
    
    print(f"Container Count: {container_repo.count()}\n")

--- CREATE TYPES & CONTAINERS ---
Created Types: Blood, Saliva
Created Container: C-001 (Cryotube)

--- QUERY TYPES ---
get_by_name('Blood'): ID 1
Container Count: 1



## 4. Protocol Repository Tests

In [7]:
# Test ProtocolRepository queries and eager loading
with uow_factory.create() as uow:
    protocol_repo = ProtocolRepository(uow.session)

    print("--- CREATE PROTOCOLS ---")
    prot1 = Protocol(code="PR-001", name="DNA Extraction", description="Standard method")
    prot2 = Protocol(code="PR-002", name="RNA Isolation", description="Advanced method")
    
    protocol_repo.save(prot1)
    protocol_repo.save(prot2)
    # Assign a reviewer to prot1
    staff_repo = StaffRepository(uow.session)
    reviewer = staff_repo.get_by_code("S-001")
    if reviewer:
        prot1.reviewed_by_id = reviewer.id
    uow.commit()
    print(f"Created Protocol: {prot1.code} - {prot1.name}")
    print(f"Created Protocol: {prot2.code} - {prot2.name}\n")

    print("--- SPECIFIC QUERIES ---")
    found_prot = protocol_repo.get_by_code("PR-001")
    print(f"get_by_code('PR-001'): {found_prot.name}")
    
    search_results = protocol_repo.search_by_name("DNA")
    print(f"search_by_name('DNA'): Found {len(search_results)} match(es)")
    
    unreviewed = protocol_repo.get_unreviewed()
    print(f"get_unreviewed(): {len(unreviewed)} protocols pending review")

--- CREATE PROTOCOLS ---
Created Protocol: PR-001 - DNA Extraction
Created Protocol: PR-002 - RNA Isolation

--- SPECIFIC QUERIES ---
get_by_code('PR-001'): DNA Extraction
search_by_name('DNA'): Found 1 match(es)
get_unreviewed(): 1 protocols pending review


In [ ]:
uow_factory.engine.dispose()